
## 1. Gussian Noise Layer:
We define a custom PyTorch layer that adds Gaussian noise to its input during training. This can help improve model robustness and prevent overfitting.

When training
$$
\begin{aligned}
\tilde{x} &= x + \epsilon \\
\epsilon &\sim \mathcal{N}(0,\sigma^2 I)
\end{aligned}
$$

Where:

- $\tilde{x}$: the noisy input (`noisy_x`)
- $\epsilon$: a noise vector, i.i.d. across dimensions
- $\sigma=\text{std}$: noise level (e.g., 0.05 or 0.03 in your code)
- $I$: identity matrix (independent dimensions with equal variance)

No noise is added:

$$
\tilde{x} = x
$$

In training, you are effectively minimizing:

$$
\min_{\theta,\phi}\ \mathbb{E}_{\epsilon}\Big[
\lVert x - g_{\phi}(f_{\theta}(x+\epsilon)) \rVert_2^2
\Big]
$$

Meaning: given a corrupted input $x+\epsilon$, the model must reconstruct the clean input $x$.

---

## 2. Autoencoder 

### (a) Encoder / Decoder / Autoencoder 

Let the input be:
$$
x \in \mathbb{R}^{d}.
$$

### (1) Denoising (training only)
During training, add Gaussian noise:
$$
\tilde{x} = x + \epsilon,
\qquad
\epsilon \sim \mathcal{N}(0,\sigma^2 I).
$$
During inference:
$$
\tilde{x} = x.
$$

### (2) Encoder: $f_\theta$
The encoder maps input to a latent vector:
$$
z = f_\theta(\tilde{x}) \in \mathbb{R}^{k}.
$$
A layer-wise view (with ReLU and Dropout):
$$
\begin{aligned}
h_1 &= \mathrm{Dropout}(\mathrm{ReLU}(W_1 \tilde{x} + b_1)), \\
h_2 &= \mathrm{Dropout}(\mathrm{ReLU}(W_2 h_1 + b_2)), \\
z   &= W_3 h_2 + b_3.
\end{aligned}
$$

### (3) Decoder: $g_\phi$
The decoder reconstructs back to input space:
$$
\hat{x} = g_\phi(z) \in \mathbb{R}^{d}.
$$
Layer-wise:
$$
\begin{aligned}
g_1 &= \mathrm{ReLU}(V_1 z + c_1), \\
g_2 &= \mathrm{ReLU}(V_2 g_1 + c_2), \\
\hat{x} &= V_3 g_2 + c_3.
\end{aligned}
$$

### (4) Reconstruction objective (MSE)
A standard AE minimizes reconstruction error:
$$
\min_{\theta,\phi}\ \mathbb{E}\left[\lVert x - \hat{x}\rVert_2^2\right].
$$
With denoising:
$$
\min_{\theta,\phi}\ \mathbb{E}_{\epsilon}\left[\left\lVert x - g_\phi\!\left(f_\theta(x+\epsilon)\right)\right\rVert_2^2\right].
$$

## 3 .Anomaly Detector (AE + Robust Gaussian (MCD) + Shewhart/EWMA)

### (a) EWMA Calculation
Given a sequence $(x_1, \dots, x_T)$, EWMA produces $(z_t)$ via


$$
z_t = \lambda x_t + (1-\lambda) z_{t-1}, \qquad t=2,\dots,T .
$$


with initialization

$$
z_1 =
\begin{cases}
\lambda x_1 + (1-\lambda) z_0, & \text{if an initial state } z_0 \text{ is provided},\\
x_1, & \text{otherwise}.
\end{cases}
$$

### (B) Fitting procedure 

(1) Autoencoder training: 

For each window vector $(x \in \mathbb{R}^{p})$, the autoencoder computes
$$
z=f_\theta(x), \qquad \hat{x}=g_\phi(z),
$$
and is trained by minimizing the reconstruction MSE:
$$
\min_{\theta,\phi}\; \frac{1}{N}\sum_{i=1}^{N}\lVert x_i-\hat{x}_i\rVert_2^2.
$$

(2) Reconstruction error (per-sample):

We define the reconstruction error as
$$
re_i = \frac{1}{p}\sum_{j=1}^{p}(x_{ij}-\hat{x}_{ij})^2,
\qquad
\tilde{re}_i = \log(1+re_i).
$$

(3) Robust latent modeling via MCD :

Given latent vectors $(\{z_i\}_{i=1}^N)$, MCD estimates a robust location and covariance
$((\hat{\mu},\hat{\Sigma}))$ by selecting a subset $(H)$ $(size (h))$ whose covariance determinant is minimized:
$$
H^* = \arg\min_{|H|=h}\det(\Sigma(H)),\quad
\hat{\mu}=\mu(H^*),\quad
\hat{\Sigma}=\Sigma(H^*).
$$

(4) Mahalanobis distance score:

The (squared) Mahalanobis distance is
$$
d_i^2 = (z_i-\hat{\mu})^\top \hat{\Sigma}^{-1} (z_i-\hat{\mu}),
\qquad
d_i=\sqrt{d_i^2}.
$$

(5) UCL thresholds via empirical quantiles:

“Set UCLs by empirical quantiles to control the in-control exceedance rate (approx. false alarm probability).
Under independence, this corresponds to ARL0 ≈ 1/(1-q), but in practice ARL0 may differ due to window overlap and EWMA dependence.”

Using the training distribution, we set upper control limits (UCLs) by the $(q)$-quantile:
$$
\mathrm{UCL}_d=\mathrm{Quantile}_q(\{d_i\}),
\quad
\mathrm{UCL}_{ew}=\mathrm{Quantile}_q(\{\mathrm{EWMA}(d_i)\}),
\quad
\mathrm{UCL}_{re}=\mathrm{Quantile}_q(\{\tilde{re}_i\}).
$$

### (c) Scoring procedure

Given scores $((d_t, ew_t, \tilde{re}_t))$, define three alarm indicators:
$$
S_t=\mathbb{1}(d_t>\mathrm{UCL}_d),\quad
E_t=\mathbb{1}(ew_t>\mathrm{UCL}_{ew}),\quad
R_t=\mathbb{1}(\tilde{re}_t>\mathrm{UCL}_{re}).
$$

We combine them using one of the following decision rules:

- OR rule:
$$
\hat{y}_t=\mathbb{1}(S_t \lor E_t \lor R_t).
$$

- AND rule:
$$
\hat{y}_t=\mathbb{1}(S_t \land E_t \land R_t).
$$

- 2-out-of-3 voting:
$$
\hat{y}_t=\mathbb{1}(S_t+E_t+R_t \ge 2).
$$



### Latent representation and Mahalanobis distance

$$
z = f_{\theta}(x), \qquad z \in \mathbb{R}^{k}.
$$

$$
d^2 = (z-\mu)^\top \Sigma^{-1}(z-\mu).
$$

Let the inverse covariance be factored via Cholesky:

$$
\Sigma^{-1} = LL^\top,
$$

and define

$$
u = L(z-\mu).
$$

Then

$$
d^2 = u^\top u = \sum_{j=1}^{k} u_j^2,
\qquad
\mathrm{contrib}_j = u_j^2.
$$

### Gradient-based input saliency

Note that gradient-based importance measures local sensitivity of the encoder and should be interpreted as saliency rather than causality. We add a small diagonal jitter 
𝜀
𝐼
εI before Cholesky decomposition to ensure positive definiteness and numerical stability.”

$$
\mathrm{importance}_m
=
\frac{1}{k}\sum_{j=1}^{k}
\left|\frac{\partial z_j}{\partial x_m}\right|,
\qquad m=1,\ldots,d.
$$


## Windowing (make_windows)

“make_windows converts a multivariate time series into overlapping flattened windows (and labels a window as anomalous if any point inside is anomalous), while window_to_point maps window-level scores back to point-level scores by aggregating the scores of all windows covering each time step (using max or mean).”

fixed-length sliding window

Let $X \in \mathbb{R}^{T \times D}$ be a multivariate time series, and (optionally) point labels
$y \in \{0,1\}^{T}$. Let the window length be $W$ and stride be $s$.

The set of window start indices is
$$
\mathcal{S}=\{0, s, 2s, \dots, T-W\}, \qquad N = |\mathcal{S}|.
$$

For the $k$-th window (with start $s_k = ks$), the windowed segment is
$$
X^{(k)} = X_{\,s_k:s_k+W-1,\; :}\in \mathbb{R}^{W\times D}.
$$

Flatten (vectorize) the window into a single vector:
$$
x^{(k)} = \mathrm{vec}\!\left(X^{(k)}\right) \in \mathbb{R}^{WD}.
$$

The center index of the window is
$$
c_k = s_k + \left\lfloor \frac{W}{2} \right\rfloor.
$$

If point labels $y$ are given, the window label is defined by
$$
y^{(w)}_k = \max_{t=s_k}^{s_k+W-1} y_t \in \{0,1\}.
$$


## Mapping window scores back to point scores (window_to_point)

Let window-level scores be $\{a_k\}_{k=0}^{N-1}$ (e.g., predicted labels or anomaly scores).
Define the coverage set for a time index $t$ as
$$
\mathcal{K}(t)=\{\,k \mid s_k \le t \le s_k+W-1 \,\}.
$$

### Max aggregation (mode = "max")
$$
A_t = \max_{k \in \mathcal{K}(t)} a_k, \qquad t=0,1,\dots,n-1.
$$

### Mean aggregation (mode = "mean")
$$
A_t =
\frac{1}{|\mathcal{K}(t)|}\sum_{k \in \mathcal{K}(t)} a_k,
\qquad t=0,1,\dots,n-1.
$$


### Fused anomaly score (normalized exceedance)

We have three monitoring statistics at time $(t)$:

- Mahalanobis distance (Shewhart): $(d_t)$
- EWMA of Mahalanobis distance: $(\mathrm{ew}_t)$
- Reconstruction error: $(\mathrm{re}_t)$

Each statistic has its own upper control limit (UCL) estimated from training data:

$$
\mathrm{UCL}_d,\quad \mathrm{UCL}_{\mathrm{ew}},\quad \mathrm{UCL}_{\mathrm{re}}.
$$

To make them comparable, we normalize each statistic by its UCL:

$$
s_t = \frac{d_t}{\mathrm{UCL}_d+\varepsilon},
\qquad
e_t = \frac{\mathrm{ew}_t}{\mathrm{UCL}_{\mathrm{ew}}+\varepsilon},
\qquad
r_t = \frac{\mathrm{re}_t}{\mathrm{UCL}_{\mathrm{re}}+\varepsilon}.
$$

Finally, we fuse them by taking the maximum (late fusion):

$$
\mathrm{score}_t = \max\{s_t,\ e_t,\ r_t\}.
$$

**Interpretation.**
If $(\mathrm{score}_t > 1)$, then at least one statistic exceeds its own UCL, so time $(t)$ is considered anomalous.
The constant $(\varepsilon)$ (e.g., $(10^{-12})$) is a numerical stabilizer to avoid division by zero.

### Point-adjusted prediction (event/segment-level evaluation)

Let the ground-truth labels be $(y_t \in \{0,1\})$ and the raw predictions be $(\hat{y}_t \in \{0,1\})$, for $(t=1,\dots,T\\)$.
Assume the positive labels form contiguous anomaly segments
$(\{[a_j,b_j)\}_{j=1}^J)$, where $(y_t=1)$ for $(t\in[a_j,b_j))$.

**Point Adjustment (PA)** declares a whole segment as detected if the model predicts at least one positive point inside that segment:

$$
\text{hit}_j =
\mathbb{I}\left(\sum_{t=a_j}^{b_j-1} \hat{y}_t > 0\right),
$$

and the adjusted predictions are defined by:

$$
\hat{y}^{\mathrm{PA}}_t =
\begin{cases}
1, & \exists j \text{ such that } t\in[a_j,b_j) \ \text{and}\ \text{hit}_j=1,\\[4pt]
\hat{y}_t, & \text{otherwise}.
\end{cases}
$$

**Interpretation.**
If a segment is “hit” at least once, PA marks all points in that segment as correctly detected.
This often increases recall substantially, and it measures segment-level detection rather than point-wise accuracy.

## results 

<figure style="text-align:center; margin: 0;">
  <img src="best3_plot/trainval.png" style="width:800px; max-width:100%; border-radius:10px;">
  <figcaption style="margin-top:8px; font-size:0.95em; line-height:1.55; text-align:left; max-width:800px; margin-left:auto; margin-right:auto; color: currentColor; opacity: 0.85;">
    <b>Figure 1. Training and Validation Loss Curves (MSE).</b>
    This figure shows the mean squared error (MSE) reconstruction loss across epochs for both the training set (blue) and validation set (orange). Overall, both curves decrease quickly in the early epochs and then gradually flatten, indicating that the model successfully learns a stable representation of normal patterns. The validation loss tracks the training loss without a persistent upward divergence, which suggests limited overfitting. A few sharp spikes in the training loss appear mid-training; these are commonly caused by stochastic mini-batch variation, occasional difficult batches, or optimization noise. Despite these spikes, the validation curve remains relatively smooth and continues improving slightly, implying that training is stable and the final checkpoint is reasonable
  </figcaption>
</figure>


<figure style="text-align:center; margin: 0;">
  <img src="best3_plot/cm_windowbase.png" style="width:800px; max-width:100%; border-radius:10px;">
  <figcaption style="margin-top:8px; font-size:0.95em; color: currentColor; opacity: 0.85; line-height:1.5; text-align:left; max-width:800px; margin-left:auto; margin-right:auto;">
    <b>Figure 2. Confusion Matrix – Window-based Detection (VAL).</b>
    This confusion matrix summarizes window-level classification on the validation set:

TN (Normal → Normal) = 6612

FP (Normal → Anomaly) = 1375

FN (Anomaly → Normal) = 9

TP (Anomaly → Anomaly) = 548

The model achieves very high recall at the window level (very few false negatives: FN=9), meaning it rarely misses true anomaly windows. However, the number of false positives is relatively large (FP=1375), implying frequent false alarms on normal windows. This pattern is typical of a sensitive detection threshold: the detector prefers catching anomalies (low FN) at the cost of triggering more alarms (high FP).“Under point-adjusted evaluation, anomaly segments are counted as detected if any window within the segment is flagged; thus PA converts FN from 9 to 0 and yields PA-Recall = 1.000.”
  </figcaption>
</figure>

<figure style="text-align:center; margin: 0;">
  <img src="best3_plot/sere.png" style="width:800px; max-width:100%; border-radius:10px;">
  <figcaption style="margin-top:8px; font-size:0.95em; color: currentColor; opacity: 0.85; line-height:1.5; text-align:left; max-width:800px; margin-left:auto; margin-right:auto;">
    <b>Figure 3. SPC-style Charts on Validation Set (Shewhart MD, EWMA, Reconstruction Error).</b>
    This figure provides three anomaly score time series with corresponding control limits (red dashed UCL lines). The shaded region highlights a known anomaly segment (or an interval of interest). An alarm is triggered when the score exceeds its UCL.

(a) Shewhart Chart – Mahalanobis Distance (MD)

The MD score measures how far each window’s latent representation deviates from the learned “normal” distribution. Most normal regions stay well below the UCL, while several bursts exceed the UCL, especially around the shaded anomaly interval. Shewhart-style monitoring is typically sensitive to abrupt spikes or sudden distribution shifts, which is consistent with the sharp peaks observed.

Interpretation: Strong peaks above UCL indicate windows whose latent patterns are highly unusual compared to normal behavior.

(b) EWMA Chart (λ = 0.2)

EWMA smooths the underlying MD-related signal to emphasize persistent or gradually developing shifts rather than single isolated spikes. With λ=0.2, the chart reacts moderately fast while still filtering noise. In your plot, the EWMA curve rises and remains elevated during the anomaly interval, crossing the UCL more continuously than Shewhart in some areas.

Interpretation: EWMA confirms that the deviation is not just a single spike; it reflects sustained abnormal behavior across consecutive windows.

(c) Reconstruction Error (RE) Chart

Reconstruction error quantifies how well the autoencoder reconstructs the input window. Normal patterns are typically reconstructed with small error (below UCL), while anomalies produce larger errors due to unseen or hard-to-model patterns. In your figure, the RE score exhibits large excursions above the UCL, particularly within (and near) the shaded anomaly region.

Interpretation: High reconstruction error suggests the observed pattern cannot be well explained by the normal manifold learned by the model.

Although the plotted curves are not classical control charts on raw process measurements, they are control-chart-inspired monitoring charts applied to learned anomaly statistics. Specifically, we monitor (i) the Mahalanobis distance in latent space with a Shewhart-style upper control limit (UCL), (ii) an EWMA-smoothed version of the score with smoothing parameter 
𝜆
λ, and (iii) the reconstruction error from the autoencoder. The UCLs are empirically calibrated using the distribution of normal training data (quantile-based limits), enabling threshold-based statistical process monitoring of anomaly scores.
  </figcaption>
</figure>


<figure style="text-align:center; margin: 0;">
  <img src="best3_plot/impactq.png" style="width:800px; max-width:100%; border-radius:10px;">
  <figcaption style="margin-top:8px; font-size:0.95em; line-height:1.55; text-align:left; max-width:800px; margin-left:auto; margin-right:auto; color: currentColor; opacity: 0.85;">
    <b>Figure 4. Impact of the UCL quantile q on point-adjusted detection performance (VAL).</b>
    This bar chart evaluates how the choice of the upper control limit (UCL) quantile q affects point-adjusted (PA) Precision, Recall, and F1-score on the validation set. Increasing 
q makes the UCL more conservative (higher threshold), which typically reduces the number of triggered alarms. In our results, PA-Recall remains at 1.0 for all tested q, indicating that every anomaly segment is still detected at least once regardless of the threshold. Meanwhile, PA-Precision increases monotonically as q increases (from ~0.27 at 𝑞=0.990q=0.990 to ~0.37 at 𝑞=0.999q=0.999), suggesting fewer false alarms on normal segments. As a result, PA-F1 also improves with larger q, reflecting a better precision–recall balance under event-level (segment-based) evaluation. Overall, this analysis shows that using a higher quantile (e.g., 𝑞=0.997,q=0.997–0.999) can significantly reduce false positives while preserving near-perfect event detection in PA evaluation.
  </figcaption>
</figure>

<figure style="text-align:center; margin: 0;">
  <img src="best3_plot/pca.png" style="width:800px; max-width:100%; border-radius:10px;">
  <figcaption style="margin-top:8px; font-size:0.95em; line-height:1.55; text-align:left; max-width:800px; margin-left:auto; margin-right:auto; color: currentColor; opacity: 0.85;">
    <b>Figure 5.Latent Space Visualization (PCA Projection).</b>
    This figure visualizes the learned latent representations by projecting them onto the first two principal components (PC1 and PC2). Each point corresponds to one window, colored by the ground-truth label (Normal vs. Anomaly). The plot shows a clear separation trend: most normal windows form a compact cluster on the right side of the space, while anomalous windows scatter into distinct regions, including a dense anomaly cluster in the upper-left area. The explained variance ratios (e.g., PC1 ≈ 79.5% and PC2 ≈ 13.5%) indicate that these two components capture most of the variability in the latent space, so the separation is meaningful rather than a projection artifact. A small number of anomaly points appear closer to the normal cluster, suggesting borderline anomalies or transitional windows that are harder to distinguish. Overall, this visualization confirms that the autoencoder effectively learns a latent representation that discriminates normal and anomalous patterns, facilitating subsequent anomaly detection based on distance metrics like Mahalanobis distance.
  </figcaption>
</figure>

<figure style="text-align:center; margin: 0;">
  <img src="best3_plot/re.png" style="width:800px; max-width:100%; border-radius:10px;">
  <figcaption style="margin-top:8px; font-size:0.95em; line-height:1.55; text-align:left; max-width:800px; margin-left:auto; margin-right:auto; color: currentColor; opacity: 0.85;">
    <b>Figure 6. Reconstruction Examples (Original vs. Reconstructed Signals).</b>
    This figure compares the original input signals (solid line) and the autoencoder reconstructions (dashed line) for several representative samples. Across samples, the reconstruction captures the dominant periodic patterns and overall waveform structure reasonably well, indicating that the model has learned the typical “normal” dynamics. However, noticeable discrepancies occur around sharp spikes and abrupt changes, where the reconstructed signal smooths or underestimates the peak magnitude. This behavior is expected in reconstruction-based anomaly detection: patterns that deviate from the learned normal manifold are harder to reconstruct accurately, leading to a larger reconstruction error (RE) and higher anomaly scores. The examples illustrate how the autoencoder effectively models regular patterns while struggling with anomalous or rare events, which is the basis for using reconstruction error as an anomaly detection signal.
  </figcaption>
</figure>


<figure style="margin:0;">
  <div style="display:flex; gap:14px; justify-content:center; flex-wrap:wrap;">
    <img src="best3_plot/base_fuse.png"
         style="width:49%; min-width:340px; max-width:100%; border-radius:10px;">
    <img src="best3_plot/best_fuse.png"
         style="width:49%; min-width:340px; max-width:100%; border-radius:10px;">
  </div>

  <figcaption style="margin-top:10px; font-size:0.95em; line-height:1.55; text-align:left; max-width:980px; margin-left:auto; margin-right:auto; color:currentColor; opacity:0.88;">
    <b>Figure 7. Fused score (Baseline vs. Best).</b> 
    The baseline histogram shows the fused anomaly-score distributions for Normal (blue) and Anomaly (orange) windows on the FINAL set. Most normal windows are concentrated near very small scores close to zero, indicating that the detector assigns low anomaly confidence to typical behavior. Anomalous windows tend to appear more in the right tail (higher scores), which supports the usefulness of the fused score for separating abnormal events. However, there is still a noticeable overlap region at low-to-mid scores, meaning some normal windows receive moderately high scores (potential false alarms) and some anomaly windows remain near the normal range (potential misses). The extremely long right tail suggests a small number of windows produce very large scores, likely corresponding to strong anomaly episodes.
   </figcaption> 
    <figcaption style="margin-top:10px; font-size:0.95em; line-height:1.55; text-align:left; max-width:980px; margin-left:auto; margin-right:auto; color:currentColor; opacity:0.88;">
    The “best” configuration histogram shows the fused-score distribution after tuning (e.g., selecting q, λ, and fusion logic based on validation). The normal distribution remains sharply peaked near zero, while anomaly windows still populate the larger-score region. Compared to the baseline, the tuned configuration aims to improve score calibration by reducing spurious high scores on normal windows and/or pushing anomalous windows toward higher scores. If the orange tail becomes more separated from the blue mass (i.e., less overlap), this indicates improved practical detectability because a single threshold can achieve better precision at similar recall.
   </figcaption> 
    <figcaption style="margin-top:10px; font-size:0.95em; line-height:1.55; text-align:left; max-width:980px; margin-left:auto; margin-right:auto; color:currentColor; opacity:0.88;">
    Comparing the two panels, the main goal is to increase separation between the normal and anomaly score distributions. A better configuration is reflected by (i) fewer normal windows in the mid/high-score region (lower false-positive tendency) and (ii) anomalies maintaining a clear high-score tail (preserved sensitivity). This distribution-level comparison complements Precision/Recall/F1 by explaining why tuning and fusion logic changes improve detection behavior.
  </figcaption>
</figure>



<figure style="text-align:center; margin: 0;">
  <img src="best3_plot/serefinal.png" style="width:800px; max-width:100%; border-radius:10px;">
  <figcaption style="margin-top:8px; font-size:0.95em; line-height:1.55; text-align:left; max-width:800px; margin-left:auto; margin-right:auto; color: currentColor; opacity: 0.85;">
    <b>Figure 8. SPC-style anomaly scores on the FINAL set (Shewhart MD, EWMA, and Reconstruction Error)..</b>
    This figure plots three anomaly-related scores over time (sample index) and applies an upper control limit (UCL, red dashed line) to trigger alarms.

Top: Shewhart chart using Mahalanobis Distance (MD). MD measures how far each window’s latent representation deviates from the “normal” distribution. Spikes above the UCL indicate windows that are statistically unlikely under the baseline normal manifold.

Middle: EWMA chart (λ = 0.1). EWMA smooths the score sequence to emphasize sustained shifts rather than isolated spikes. Compared with the Shewhart chart, EWMA is generally more sensitive to gradual changes and persistent anomaly regimes.

Bottom: Reconstruction Error (RE). RE reflects how well the autoencoder reconstructs the input window. Larger errors suggest the pattern is not well explained by the learned normal behavior.

The highlighted/shaded regions indicate ground-truth anomaly intervals (or detected anomaly segments, depending on your plotting). Across these intervals, all three scores show elevated activity and/or repeated UCL crossings, suggesting consistent anomaly evidence from complementary perspectives: distributional deviation (MD), persistent shift (EWMA), and reconstruction difficulty (RE).
    </figcaption>
</figure>

<figure style="margin:0;">
  <div style="display:flex; gap:14px; justify-content:center; flex-wrap:wrap;">
    <img src="best3_plot/basecm.png"
         style="width:49%; min-width:340px; max-width:100%; border-radius:10px;">
    <img src="best3_plot/bestcm.png"
         style="width:49%; min-width:340px; max-width:100%; border-radius:10px;">
  </div>

  <figcaption style="margin-top:10px; font-size:0.95em; line-height:1.55; text-align:left; max-width:980px; margin-left:auto; margin-right:auto; color:currentColor; opacity:0.88;">
    <b>Figure 9. Confusion matrix (Baseline vs. Best).</b> 
    his confusion matrix corresponds to the baseline setting (q=0.997, λ=0.2, OR fusion). Counts are:

TN = 15,572

FP = 2,227

FN = 158

TP = 1,979

The baseline OR logic is recall-oriented: it flags an anomaly if any detector triggers, which yields many true positives and very few missed anomalies (low FN). However, this comes with a substantial increase in false positives (high FP), meaning the system will produce many false alarms during normal periods.
   </figcaption> 
    <figcaption style="margin-top:10px; font-size:0.95em; line-height:1.55; text-align:left; max-width:980px; margin-left:auto; margin-right:auto; color:currentColor; opacity:0.88;">
    This confusion matrix summarizes window-level classification under the tuned setting (q=0.999, λ=0.10, vote). The matrix shows:

TN = 17,445 (normal correctly predicted as normal)

FP = 354 (normal incorrectly flagged as anomaly)

FN = 1,322 (anomaly predicted as normal)

TP = 815 (anomaly correctly flagged)

This configuration is conservative: it dramatically reduces false alarms (low FP) by requiring stronger and more consistent evidence (via voting) before declaring anomalies. The trade-off is a higher number of missed anomaly windows (FN) at the strict window-level evaluation. In practice, this setting is often preferred when false alarms are expensive, and the evaluation also includes event-level metrics (e.g., point-adjusted) where detecting an anomaly segment at least once can still be counted as success.
   </figcaption> 
    <figcaption style="margin-top:10px; font-size:0.95em; line-height:1.55; text-align:left; max-width:980px; margin-left:auto; margin-right:auto; color:currentColor; opacity:0.88;">
    Comparing the two confusion matrices reveals a clear precision–recall trade-off driven by fusion strictness. The baseline OR rule is highly sensitive and minimizes missed anomalies (FN=158) but produces many false alarms (FP=2,227). In contrast, the best voting-based configuration significantly suppresses false alarms (FP=354) but sacrifices window-level recall (FN=1,322). This pattern is consistent with a monitoring setting: OR behaves like a “high-sensitivity alarm,” while vote (with a stricter UCL and smaller EWMA λ) behaves like a “high-specificity alarm.” Which one is “better” depends on whether the objective prioritizes event coverage (favor OR) or alarm reliability (favor vote/strict fusion), especially when point-adjusted evaluation is considered.
  </figcaption>
</figure>


<figure style="text-align:center; margin: 0;">
  <img src="best3_plot/serefull.png" style="width:800px; max-width:100%; border-radius:10px;">
  <figcaption style="margin-top:8px; font-size:0.95em; line-height:1.55; text-align:left; max-width:800px; margin-left:auto; margin-right:auto; color: currentColor; opacity: 0.85;">
    <b>Figure 10. SPC-style charts on the FULL TEST set (MD Shewhart, EWMA, Reconstruction Error)</b>
    This figure visualizes three anomaly evidence signals across the full test timeline (x-axis = sample index). The red dashed line in each panel is an upper control limit (UCL) derived from a high quantile threshold (or an equivalent rule), and the light blue shaded bands indicate labeled anomaly intervals (or segments of interest, depending on your plotting).

Shewhart chart (Mahalanobis Distance, MD).
The MD score measures how far each window’s latent representation deviates from the normal reference distribution. Most of the time, the MD line stays well below the UCL, indicating stable normal behavior. During several shaded anomaly intervals, MD exhibits sharp spikes and frequent UCL exceedances, suggesting strong distributional shifts relative to normal patterns. Occasional isolated spikes outside shaded regions may correspond to borderline abnormal windows or valid-but-rare normal fluctuations.

EWMA chart (λ = 0.2).
EWMA smooths the score sequence and emphasizes sustained changes. Compared with the Shewhart MD chart, EWMA tends to rise and remain elevated during prolonged abnormal regimes, which helps detect gradual drift or persistent anomalies rather than single outliers. In the middle portion of the timeline (around the major anomaly region), EWMA shows the most pronounced elevation and repeated UCL crossings, consistent with long-lasting abnormal behavior.

Reconstruction Error (RE).
Reconstruction error captures how well the autoencoder reconstructs the input window. The RE curve stays near a low baseline for most normal periods, but becomes noticeably larger in anomaly bands, indicating that abnormal patterns are not well explained by the learned normal manifold. Compared to MD/EWMA, RE peaks can be more localized, reflecting windows where the signal shape deviates strongly from training-like patterns.

Overall, the three panels provide complementary evidence: MD detects distributional deviation, EWMA highlights sustained shifts, and RE reflects representation mismatch. Their agreement (simultaneous increases/UCL hits) strengthens confidence that the highlighted periods are genuinely anomalous.
    </figcaption>
</figure>




<figure style="text-align:center; margin: 0;">
  <img src="best3_plot/fullce.png" style="width:800px; max-width:100%; border-radius:10px;">
  <figcaption style="margin-top:8px; font-size:0.95em; line-height:1.55; text-align:left; max-width:800px; margin-left:auto; margin-right:auto; color: currentColor; opacity: 0.85;">
    <b>Figure 11. Confusion Matrix (Sanity Check – det_base on FULL TEST)</b>
    This confusion matrix summarizes window-level predictions for the baseline detector on the FULL TEST set:

TN = 24,960 (normal correctly predicted as normal)

FP = 826 (normal incorrectly flagged as anomaly)

FN = 1,034 (anomaly predicted as normal)

TP = 1,660 (anomaly correctly flagged)

Interpretation:

The detector correctly classifies the majority of normal windows (large TN), with a moderate number of false alarms (FP).

It also detects a substantial portion of anomaly windows (TP), but still misses some anomalies (FN), which is common when anomalies vary in severity and not every anomalous window produces a strong score peak.

This “sanity check” is consistent with the time-series plots: large anomaly regimes trigger many alarms, while weaker/shorter anomalies may not always cross the UCL.
    </figcaption>
</figure>



| Setting | q | λ | logic | P | R | F1 |
|---|---:|---:|---|---:|---:|---:|
| Baseline | 0.997 | 0.20 | or   | 0.471 | 0.926 | 0.624 |
| Best     | 0.997 | 0.20 | vote | 0.946 | 0.642 | 0.764 |

**Training summary**
| Item | Value |
|---|---:|
| Final Train Loss (MSE) | 0.041257 |
| Final Valid Loss (MSE) | 0.047844 |
| Best Valid Loss (MSE) | 0.047844 |
| Best Epoch | 60 |

**Window-level metrics (VAL)**
| Metric | Original | Point-Adjusted (PA) | Δ (PA − Original) |
|---|---:|---:|---:|
| Accuracy | 0.838 | 0.839 | +0.001 |
| Precision | 0.285 | 0.288 | +0.003 |
| Recall | 0.984 | 1.000 | +0.016 |
| F1-score | 0.442 | 0.448 | +0.006 |
| ROC-AUC | 0.943 *(from score)* | 0.914 *(from PA 0/1 preds)* | — |
| PR-AUC | 0.661 *(from score)* | — | — |

**Confusion matrix (Original, window-level)**
|  | Pred Normal | Pred Anomaly |
|---|---:|---:|
| True Normal | 6612 (TN) | 1375 (FP) |
| True Anomaly | 9 (FN) | 548 (TP) |

**Confusion matrix (Point-Adjusted, window-level)**
|  | Pred Normal | Pred Anomaly |
|---|---:|---:|
| True Normal | 6612 | 1375 |
| True Anomaly | 0 | 557 |

**Interpretation (report-ready)**
- The detector achieves **very high recall** on validation (**Recall = 0.984**, and **PA-Recall = 1.000**), meaning it **rarely misses anomaly windows** (FN = 9; PA-FN = 0).
- However, **false alarms are relatively frequent** (**FP = 1375**), which lowers precision (**Precision ≈ 0.285–0.288**). This indicates the current threshold/logic is **sensitivity-oriented**: it prioritizes catching anomalies at the cost of more normal windows being flagged.
- Point-Adjusted evaluation improves recall to 1.000 because it treats an anomaly *segment* as correctly detected if **any point/window within the segment is detected**, which better reflects “event-level” detection capability.

### Validation Results (Window-based, Best Epoch = 60)

**Training summary**
| Item | Value |
|---|---:|
| Final Train Loss (MSE) | 0.041257 |
| Final Valid Loss (MSE) | 0.047844 |
| Best Valid Loss (MSE) | 0.047844 |
| Best Epoch | 60 |

**Window-level metrics (VAL)**
| Metric | Original | Point-Adjusted (PA) | Δ (PA − Original) |
|---|---:|---:|---:|
| Accuracy | 0.838 | 0.839 | +0.001 |
| Precision | 0.285 | 0.288 | +0.003 |
| Recall | 0.984 | 1.000 | +0.016 |
| F1-score | 0.442 | 0.448 | +0.006 |
| ROC-AUC | 0.943 *(from score)* | 0.914 *(from PA 0/1 preds)* | — |
| PR-AUC | 0.661 *(from score)* | — | — |



**Interpretation (report-ready)**
- The detector achieves **very high recall** on validation (**Recall = 0.984**, and **PA-Recall = 1.000**), meaning it **rarely misses anomaly windows** (FN = 9; PA-FN = 0).
- However, **false alarms are relatively frequent** (**FP = 1375**), which lowers precision (**Precision ≈ 0.285–0.288**). This indicates the current threshold/logic is **sensitivity-oriented**: it prioritizes catching anomalies at the cost of more normal windows being flagged.
- Point-Adjusted evaluation improves recall to 1.000 because it treats an anomaly *segment* as correctly detected if **any point/window within the segment is detected**, which better reflects “event-level” detection capability.



### Validation Study: Effect of Fusion Logic (OR vs. VOTE vs. AND)

To combine multiple detection signals (e.g., Shewhart/MD, EWMA, and reconstruction error), we compared three fusion logics on the validation set under **point-adjusted (PA)** evaluation.

**Results (PA metrics on VAL)**

| Fusion logic | Decision rule | PA Precision | PA Recall | PA F1 |
|---|---|---:|---:|---:|
| OR   | Alarm if **any** detector triggers | 0.288 | 1.000 | 0.448 |
| VOTE | Alarm if **majority** triggers | 0.423 | 0.995 | 0.593 |
| AND  | Alarm if **all** detectors trigger | 0.518 | 0.995 | 0.681 |


**OR** is the most sensitive rule: it achieves the highest recall (PA-Recall = 1.000) but suffers from low precision because many alarms are caused by a single noisy detector.

**VOTE** provides a better balance by requiring agreement from multiple detectors, which reduces false alarms while keeping recall near perfect.

**AND** is the most conservative rule and yields the **best overall PA-F1** (0.681). Precision improves substantially while recall remains ~1.0, indicating that consensus across detectors effectively suppresses false positives without sacrificing event-level detection coverage.

**Key takeaway:** As the fusion rule becomes stricter (OR → VOTE → AND), **precision increases** while **PA-recall stays nearly unchanged**, and **AND** gives the best trade-off on VAL.

**Confusion matrix (Point-Adjusted, window-level)**
|  | Pred Normal | Pred Anomaly |
|---|---:|---:|
| True Normal | 6612 | 1375 |
| True Anomaly | 0 | 557 |

## Summary Table (FINAL / FULL TEST)

> Metrics are reported for **window-level (Original)** and **point-adjusted (PA)** evaluation.

### (A) Overall metrics

| Setting | Split | q | λ | Logic | Acc (Orig) | P (Orig) | R (Orig) | F1 (Orig) | ROC-AUC | PR-AUC | Acc (PA) | P (PA) | R (PA) | F1 (PA) | ROC-AUC (PA) |
|---|---|---:|---:|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| Baseline | FINAL | 0.997 | 0.20 | OR | 0.880 | 0.471 | 0.926 | 0.624 | 0.944 | 0.663 | 0.888 | 0.489 | 0.999 | 0.657 | 0.937 |
| Best | FINAL | 0.999 | 0.10 | VOTE | 0.916 | 0.697 | 0.381 | 0.493 | 0.944 | 0.635 | 0.982 | 0.858 | 0.998 | 0.923 | 0.989 |
| Sanity Check | FULL TEST | - | - | det_base | 0.935 | 0.668 | 0.616 | 0.641 | 0.952 | 0.672 | 0.971 | 0.765 | 0.997 | 0.866 | 0.983 |

### B Confusion matrices (Original, window-level)

**Baseline on FINAL (q=0.997, λ=0.2, logic=OR)**  
| Item| Pred Normal | Pred Anomaly |
|---|---:|---:|
| True Normal | 15572 (TN) | 2227 (FP) |
| True Anomaly | 158 (FN) | 1979 (TP) |

**Best on FINAL (q=0.999, λ=0.10, logic=VOTE)**  
|  Item| Pred Normal | Pred Anomaly |
|---|---:|---:|
| True Normal | 17445 (TN) | 354 (FP) |
| True Anomaly | 1322 (FN) | 815 (TP) |

**Sanity Check on FULL TEST (det_base)**  
| Item | Pred Normal | Pred Anomaly |
|---|---:|---:|
| True Normal | 24960 (TN) | 826 (FP) |
| True Anomaly | 1034 (FN) | 1660 (TP) |

### C Confusion matrices (Point-Adjusted, PA)

**Baseline on FINAL (PA)**  
| Item | Pred Normal | Pred Anomaly |
|---|---:|---:|
| True Normal | 15572 | 2227 |
| True Anomaly | 2 | 2135 |

**Best on FINAL (PA)**  
| Item | Pred Normal | Pred Anomaly |
|---|---:|---:|
| True Normal | 17445 | 354 |
| True Anomaly | 4 | 2133 |

**Sanity Check on FULL TEST (PA)**  
| Item | Pred Normal | Pred Anomaly |
|---|---:|---:|
| True Normal | 24960 | 826 |
| True Anomaly | 7 | 2687 |


## Interpretation (report-ready, English)

- **Window-level vs. Point-Adjusted (PA):** Window-level metrics require *every anomalous window* to be correctly flagged. PA metrics evaluate at the **event/segment level**: if an anomaly segment contains at least one detection, the segment is credited as detected. Therefore, PA recall is typically much higher.

- **Baseline (FINAL, OR fusion):** High sensitivity at the window level (**R=0.926**) but many false alarms (**FP=2227**), reflecting that OR triggers when any detector spikes.

- **Best (FINAL, VOTE fusion with stricter q and smaller λ):** Considerably fewer false alarms (**FP=354**) and much higher **PA precision/F1** (P=0.858, F1=0.923). However, the stricter fusion reduces window-level recall (**R=0.381**), meaning many anomaly windows are not individually flagged, even though anomaly segments are still detected under PA (**R≈0.998**).

- **Sanity check (FULL TEST):** The detector generalizes reasonably to the full test stream with strong discrimination (ROC-AUC=0.952, PR-AUC=0.672). PA recall remains near-perfect (**0.997**), indicating robust **event-level coverage** while keeping false alarms moderate (**FP=826**).

## Grid Search Results (VAL) — Ranked by PA-F1

We performed a grid search over the detection hyperparameters **q** (UCL quantile), **λ** (EWMA smoothing), and **fusion logic** (OR / VOTE / AND). Each configuration is evaluated on the **validation set** using both **window-level (RAW)** metrics and **point-adjusted (PA)** metrics. The ranking score is **PA-F1**.

### A Top configurations (Top 10 by PA-F1)

| Rank | q | λ | Logic | P (RAW) | R (RAW) | F1 (RAW) | P (PA) | R (PA) | F1 (PA) = Score |
|---:|---:|---:|---|---:|---:|---:|---:|---:|---:|
| 1 | 0.999 | 0.10 | VOTE | 0.7108 | 0.5296 | 0.6070 | 0.8220 | 0.9946 | **0.9001** |
| 2 | 0.999 | 0.30 | AND  | 0.6010 | 0.6517 | 0.6253 | 0.6969 | 0.9946 | 0.8195 |
| 3 | 0.999 | 0.20 | AND  | 0.5434 | 0.5278 | 0.5355 | 0.6916 | 0.9946 | 0.8159 |
| 4 | 0.999 | 0.10 | AND  | 0.5223 | 0.5476 | 0.5346 | 0.6651 | 0.9946 | 0.7971 |
| 5 | 0.997 | 0.30 | AND  | 0.5433 | 0.6535 | 0.5933 | 0.6442 | 0.9946 | 0.7819 |
| 6 | 0.997 | 0.10 | AND  | 0.5364 | 0.6355 | 0.5818 | 0.6442 | 0.9946 | 0.7819 |
| 7 | 0.995 | 0.10 | AND  | 0.5453 | 0.6912 | 0.6097 | 0.6331 | 0.9946 | 0.7737 |
| 8 | 0.995 | 0.30 | AND  | 0.5413 | 0.6948 | 0.6085 | 0.6281 | 0.9946 | 0.7700 |
| 9 | 0.997 | 0.20 | AND  | 0.5288 | 0.7092 | 0.6058 | 0.6115 | 0.9946 | 0.7573 |
| 10 | 0.995 | 0.20 | AND | 0.5090 | 0.6607 | 0.5750 | 0.6095 | 0.9946 | 0.7558 |

### B Best hyperparameter setting (selected by PA-F1)

| Score metric | q | λ | Logic | Best PA-F1 |
|---|---:|---:|---|---:|
| PA-F1 | 0.999 | 0.10 | VOTE | **0.900** |

### Interpretation (report-ready, English)

- **What q and λ control:**  
  - **q (UCL quantile)** sets how strict the anomaly threshold is (higher q → stricter threshold → fewer alarms).  
  - **λ (EWMA)** controls smoothing (smaller λ → smoother, less reactive; larger λ → more reactive).

- **Key pattern from the grid search:**  
  The top-ranked solutions are dominated by **stricter fusion rules** (mostly **AND**, with the best being **VOTE**) and **high q (0.999)**. This indicates many false positives arise from **single-detector spikes**; requiring agreement across detectors suppresses false alarms while maintaining extremely high **PA recall**.

- **Why the best is VOTE at (q=0.999, λ=0.10):**  
  This setting achieves **high PA precision (0.822)** while keeping **PA recall near 1.0**, resulting in the best overall **PA-F1 (0.900)**. In practice, it provides a strong **event-level** trade-off: it detects anomaly segments reliably without being overwhelmed by false alarms.

```mermaid
graph TB
    subgraph Data_Preprocessing["Data Preprocessing"]
        A["Raw Time Series<br/>T x D"] --> B["Standardization<br/>StandardScaler"]
        B --> C["Sliding Window<br/>win=30, stride=1"]
        C --> D["Windowed Data<br/>N x win x D"]
    end

    subgraph Feature_Learning["Feature Learning"]
        D --> E["Autoencoder<br/>input_dim -> latent_dim=6"]
        E --> F["Latent Space Z<br/>N x 6"]
        E --> G["Reconstruction Xhat<br/>N x win x D"]
    end

    subgraph Anomaly_Scoring["Anomaly Scoring"]
        F --> H["Robust Covariance<br/>MinCovDet"]
        H --> I["Mahalanobis Distance<br/>d_i^2 = (z-mu)^T Sigma^-1 (z-mu)"]
        G --> J["Reconstruction Error<br/>RE = log1p( norm2(X - Xhat) )"]
    end

    subgraph SPC["Statistical Process Control"]
        I --> K["Shewhart Chart<br/>UCL = quantile_q"]
        I --> L["EWMA Chart<br/>Z_t = lambda d_t + (1-lambda) Z_{t-1}"]
        J --> M["RE Chart<br/>UCL = quantile_q"]
    end

    subgraph Decision_Making["Decision Making"]
        K --> N{"Logic"}
        L --> N
        M --> N
        N -->|OR| O["Any exceeds UCL"]
        N -->|VOTE| P[">=2/3 exceed UCL"]
        N -->|AND| Q["All exceed UCL"]
        O --> R["Predictions"]
        P --> R
        Q --> R
    end

    subgraph Evaluation["Evaluation"]
        R --> S["Point-Adjusted<br/>Event-level"]
        S --> T["Metrics<br/>P, R, F1, ROC-AUC"]
    end

    style E fill:#ff9999
    style H fill:#99ccff
    style N fill:#99ff99
    style S fill:#ffcc99
``` 

```mermaid
graph LR
    subgraph ENC["Encoder"]
        A["Input<br/>win x D"] --> B["GaussianNoise<br/>std=0.03"]
        B --> C["Linear 64<br/>+ ReLU + Dropout"]
        C --> D["Linear 32<br/>+ ReLU + Dropout"]
        D --> E["Linear latent_dim<br/>Latent Space Z"]
    end

    subgraph DEC["Decoder"]
        E --> F["Linear 32<br/>+ ReLU"]
        F --> G["Linear 64<br/>+ ReLU"]
        G --> H["Linear win x D<br/>Reconstructed Xhat"]
    end

    subgraph LOSS["Loss"]
        H --> I["MSE Loss<br/>sum (X - Xhat)^2"]
        I --> J["AdamW<br/>Optimizer"]
        J --> C
    end

    style E fill:#ff9999
    style I fill:#ffcc99
``` 

```mermaid
graph TB

    subgraph P1["Phase 1: Data Preparation"]
        A1["Load Data"] --> A2["Train: Normal only<br/>Test: Normal + Anomaly"]
        A2 --> A3["Stratified Split Test<br/>VAL 30% + FINAL 70%"]
        A3 --> A4{"Anomaly Ratio Similar?"}
        A4 -->|Yes| A5["Ready"]
        A4 -->|No| A3
    end

    subgraph P2["Phase 2: Model Training"]
        A5 --> B1["Create Windows<br/>Train/VAL/FINAL"]
        B1 --> B2["Train Autoencoder<br/>on Train set"]
        B2 --> B3["Early Stopping<br/>patience=10"]
        B3 --> B4["Extract Latent Z<br/>Fit MCD"]
        B4 --> B5["Calculate UCLs<br/>Shewhart/EWMA/RE"]
    end

    subgraph P3["Phase 3: Hyperparameter Tuning"]
        B5 --> C1["Grid Search on VAL"]
        C1 --> C2["Test q in {0.99,0.995,0.997,0.999}<br/>lambda in {0.1,0.2,0.3}<br/>logic in {or,vote,and}"]
        C2 --> C3["Evaluate PA-F1<br/>on VAL"]
        C3 --> C4["Select Best Params<br/>q=0.999, lambda=0.1, logic=and"]
    end

    subgraph P4["Phase 4: Final Evaluation"]
        C4 --> D1["Apply Best Params<br/>on FINAL only once"]
        D1 --> D2["Calculate Metrics<br/>RAW + PA"]
        D2 --> D3["Generate Reports<br/>Confusion Matrix, Charts"]
        D3 --> D4["Complete"]
    end

    style A4 fill:#ffcc99
    style C4 fill:#99ff99
    style D1 fill:#ff9999

``` 

```mermaid
graph TB
    A["Start Grid Search"] --> B["Initialize Best Score = -Inf"]
    
    B --> C{"For each<br/>q, lambda, logic"}
    
    C --> D["Train New Model<br/>with params q, lambda"]
    D --> E["Predict on VAL<br/>with logic"]
    E --> F["Window -> Point<br/>Mapping"]
    F --> G["Point-Adjusted<br/>Evaluation"]
    G --> H["Calculate PA-F1"]
    
    H --> I{"PA-F1 > Best Score?"}
    I -->|Yes| J["Update Best Params<br/>Best Detector<br/>Best Score"]
    I -->|No| K["Skip"]
    
    J --> L{"More<br/>Combinations?"}
    K --> L
    
    L -->|Yes| C
    L -->|No| M["Return Best Detector<br/>Best Params<br/>Results DataFrame"]
    
    style I fill:#ffcc99
    style J fill:#99ff99
    style M fill:#99ccff


```

```mermaid
graph TB
    subgraph "Input"
        A["True Labels y_true<br/>01001110010..."]
        B["Predictions y_pred<br/>00001100000..."]
    end

    subgraph "Segment Detection"
        A --> C["Identify Anomaly<br/>Segments"]
        C --> D["Segment 1: 2-3<br/>Segment 2: 5-8"]
    end

    subgraph "Point Adjustment"
        D --> E{"For each<br/>segment"}
        E --> F{"Any point<br/>detected?"}
        F -->|Yes| G["Mark entire<br/>segment as 1"]
        F -->|No| H["Keep as 0"]
        G --> I["Adjusted Predictions<br/>y_pred_adj"]
        H --> I
    end

    subgraph "Metrics"
        I --> J["Calculate<br/>Precision, Recall, F1"]
        A --> J
        J --> K["Event-level<br/>Performance"]
    end

    style F fill:#ffcc99
    style G fill:#99ff99
    style K fill:#99ccff

```

```mermaid
graph TB
    subgraph "Core Modules"
        A['GaussianNoise<br/>Regularization']
        B['Encoder<br/>Feature Extraction']
        C['Decoder<br/>Reconstruction']
        D['Autoencoder<br/>End-to-End Model']
    end
    
    subgraph "Detection Module"
        E['AE_MCD_Detector<br/>Main Class']
        E --> E1['fit<br/>Training']
        E --> E2['score<br/>Anomaly Scoring']
        E --> E3['predict<br/>Classification']
        E --> E4['explain_anomaly<br/>Interpretability']
    end
    
    subgraph "Utility Functions"
        F['make_windows<br/>Preprocessing']
        G['window_to_point<br/>Postprocessing']
        H['point_adjust<br/>Evaluation']
        I['evaluate_and_report<br/>Metrics]
    end'
    
    subgraph "Optimization"
        J['grid_search_hyperparameters<br/>AutoML']
    end
    
    D --> E
    E --> F
    E --> G
    G --> H
    H --> I
    E --> J
    
    style E fill:#ff9999
    style J fill:#99ccff

```